In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Generic helpers
# ============================================================

def compute_fuel_share(df: pd.DataFrame,
                       fuel_col: str,
                       fuel_name: str,
                       value_col: str = "Value",
                       year_col: str = "Year") -> pd.DataFrame:
    """Compute annual share (%) of a given fuel in total generation.

    Returns
    -------
    DataFrame with columns:
        Year, Total_GWh, <fuel_name>_GWh, <fuel_name>_pct
    """
    wide = df.pivot(index=year_col, columns=fuel_col, values=value_col)
    wide["Total_GWh"] = wide.sum(axis=1)
    wide[f"{fuel_name}_GWh"] = wide[fuel_name]
    wide[f"{fuel_name}_pct"] = wide[fuel_name] / wide["Total_GWh"] * 100.0
    out = wide[["Total_GWh", f"{fuel_name}_GWh", f"{fuel_name}_pct"]].reset_index()
    out.rename(columns={year_col: "Year"}, inplace=True)
    return out


def fit_linear_trend(years: np.ndarray, values: np.ndarray, t0: int = None):
    """Fit linear trend v(t) = a + b*(t - t0).

    Returns
    -------
    (a, b, t0)
    """
    if t0 is None:
        t0 = int(years.min())
    x = years - t0
    A = np.vstack([x, np.ones_like(x)]).T
    b_slope, a_intercept = np.linalg.lstsq(A, values, rcond=None)[0]
    return float(a_intercept), float(b_slope), t0


def linear_path(v0, v1, t0, t1, years):
    """Linear interpolation / extrapolation between (t0, v0) and (t1, v1)."""
    years = np.asarray(years)
    return v0 + (v1 - v0) * (years - t0) / (t1 - t0)


def build_ssp_shares(years_historical,
                     shares_historical,
                     anchor_year,
                     ssp1_2050,
                     ssp2_2050,
                     ssp5_2050,
                     last_year=2050):
    """From a historical share series, build SSP1/SSP2/SSP5 share paths (%)."""
    hist_df = pd.DataFrame({"Year": years_historical, "Share_pct": shares_historical})
    anchor_share = float(hist_df.loc[hist_df["Year"] == anchor_year, "Share_pct"])
    future_years = np.arange(anchor_year, last_year + 1)
    ssp_defs = {
        "SSP1-RCP1.9": ssp1_2050,
        "SSP2-RCP4.5": ssp2_2050,
        "SSP5-RCP8.5": ssp5_2050,
    }
    scenarios = {}
    for ssp_name, target_2050 in ssp_defs.items():
        fut_shares = linear_path(anchor_share, target_2050,
                                 anchor_year, last_year, future_years)
        df_ssp = pd.DataFrame({"Year": future_years, "Share_pct": fut_shares})
        scenarios[ssp_name] = df_ssp
    return hist_df, scenarios


def plot_ssp_shares(hist_df, scenarios, title, ylabel, ylim=None):
    """Plot historical + SSP futures for a single marker fuel."""
    plt.figure()
    plt.plot(hist_df["Year"], hist_df["Share_pct"], "k-", label="Historical")
    for name, df_s in scenarios.items():
        plt.plot(df_s["Year"], df_s["Share_pct"], label=name)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def scenario_snapshot_table(scenarios, years=(2030, 2040, 2050)):
    """Extract a table of scenario shares (%) at selected years."""
    rows = []
    for name, df_s in scenarios.items():
        row = {"Scenario": name}
        for y in years:
            val = float(df_s.loc[df_s["Year"] == y, "Share_pct"])
            row[str(y)] = round(val, 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Scenario")


# ============================================================
# 1) Coal-dominant case (China) – historic coal% + SSP futures
# ============================================================

# Replace these file names with your own paths if needed
china_path = "e281e10a-327e-4db0-b0d2-75b34d6621a0.csv"   # China IEA CSV
canada_path = "04f7ace1-03f0-4740-8518-2230d91e2994.csv"  # Canada IEA CSV

df_china = pd.read_csv(china_path)

coal_share_ch = compute_fuel_share(
    df_china,
    fuel_col="electricity generation in China",
    fuel_name="Coal",
    value_col="Value",
    year_col="Year",
)

years_ch = coal_share_ch["Year"].to_numpy()
coal_pct_hist = coal_share_ch["Coal_pct"].to_numpy()

# Linear trend for SSP2 endpoint
a_coal, b_coal, t0_coal = fit_linear_trend(years_ch, coal_pct_hist)
coal_2050_lin = a_coal + b_coal * (2050 - t0_coal)

cp_2023 = float(coal_share_ch.loc[coal_share_ch["Year"] == 2023, "Coal_pct"])

# Assumptions for 2050 coal share (%)
ssp1_coal_2050 = 10.0          # strong mitigation
ssp2_coal_2050 = coal_2050_lin # continue historical decline
ssp5_coal_2050 = 80.0          # fossil-fuelled

hist_coal_df, coal_scenarios = build_ssp_shares(
    years_ch,
    coal_pct_hist,
    anchor_year=2023,
    ssp1_2050=ssp1_coal_2050,
    ssp2_2050=ssp2_coal_2050,
    ssp5_2050=ssp5_coal_2050,
    last_year=2050,
)

plot_ssp_shares(
    hist_coal_df,
    coal_scenarios,
    title="Coal share in coal-dominant market (China-based)",
    ylabel="Coal share of generation (%)",
    ylim=(0, 90),
)

coal_snapshot = scenario_snapshot_table(coal_scenarios)
print("Coal-dominant case – coal share (% of generation):")
print(coal_snapshot)
print()


# ============================================================
# 2) Hydro-dominant case (Canada) – historic hydro% + SSP futures
# ============================================================

df_can = pd.read_csv(canada_path)

hydro_share_ca = compute_fuel_share(
    df_can,
    fuel_col="electricity generation in Canada",
    fuel_name="Hydropower",
    value_col="Value",
    year_col="Year",
)

years_ca = hydro_share_ca["Year"].to_numpy()
hydro_pct_hist = hydro_share_ca["Hydropower_pct"].to_numpy()

a_hydro, b_hydro, t0_hydro = fit_linear_trend(years_ca, hydro_pct_hist)
hydro_2050_lin = a_hydro + b_hydro * (2050 - t0_hydro)

hp_2024 = float(hydro_share_ca.loc[hydro_share_ca["Year"] == 2024, "Hydropower_pct"])

# Assumptions for 2050 hydro share (%)
ssp1_hydro_2050 = 70.0           # sustainability: hydro remains dominant
ssp2_hydro_2050 = hydro_2050_lin # middle-of-road: similar level
ssp5_hydro_2050 = 40.0           # fossil-fuelled: hydro share erodes

hist_hydro_df, hydro_scenarios = build_ssp_shares(
    years_ca,
    hydro_pct_hist,
    anchor_year=2024,
    ssp1_2050=ssp1_hydro_2050,
    ssp2_2050=ssp2_hydro_2050,
    ssp5_2050=ssp5_hydro_2050,
    last_year=2050,
)

plot_ssp_shares(
    hist_hydro_df,
    hydro_scenarios,
    title="Hydropower share in hydro-dominant market (Canada-based)",
    ylabel="Hydropower share of generation (%)",
    ylim=(30, 75),
)

hydro_snapshot = scenario_snapshot_table(hydro_scenarios)
print("Hydro-dominant case – hydropower share (% of generation):")
print(hydro_snapshot)
